# 01 - Petrophysical Calculation for CO2 Storage Screening
### Niger Delta case study | Transferable to CaMI FRS, Alberta
**Author:** Umaru Ibe (github.com/Umaru-Ibe) | Prospective MSc Geoscience, UCalgary

Goal: Calculate Vsh, Phi_e, Perm, NTG from GR, NPHI, RHOB, Resistivity, DT logs.
This workflow is reproducible and uses Niger Delta well logs, methodology transferable to Western Canadian Sedimentary Basin.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load sample data (replace with real LAS later)
np.random.seed(42)
depth = np.arange(2000, 2600, 0.5)
gr = np.random.normal(80, 20, len(depth))
nphi = np.random.normal(0.25, 0.05, len(depth))
rhob = np.random.normal(2.2, 0.15, len(depth))
rt = np.random.normal(20, 5, len(depth))
dt = np.random.normal(90, 10, len(depth))

df = pd.DataFrame({'DEPTH':depth, 'GR':gr, 'NPHI':nphi, 'RHOB':rhob, 'RT':rt, 'DT':dt})
df.head()


In [ ]:
# Petrophysical calculations
# 1. Vshale from GR
GRmin, GRmax = 30, 120
df['VSH'] = (df['GR']-GRmin)/(GRmax-GRmin)
df['VSH'] = df['VSH'].clip(0,1)

# 2. Density porosity
RHOma, RHOfl = 2.65, 1.0
df['PHID'] = (RHOma - df['RHOB'])/(RHOma - RHOfl)

# 3. Effective porosity (simple)
df['PHIE'] = df['PHID']*(1-df['VSH']*0.3) # shale correction
df['PHIE'] = df['PHIE'].clip(0.05,0.35)

# 4. Permeability Timur-Coates
df['PERM_mD'] = 100 * (df['PHIE']**4.5) / (df['VSH']+0.1) * 10

# 5. NTG flag
df['RESERVOIR_FLAG'] = ((df['VSH']<0.4) & (df['PHIE']>0.12)).astype(int)

df.describe()


In [ ]:
# Plot
fig, ax = plt.subplots(1,4, figsize=(12,5), sharey=True)
ax[0].plot(df['GR'], df['DEPTH']); ax[0].set_xlabel('GR'); ax[0].invert_yaxis()
ax[1].plot(df['VSH'], df['DEPTH']); ax[1].set_xlabel('VSH')
ax[2].plot(df['PHIE'], df['DEPTH']); ax[2].set_xlabel('PHIE')
ax[3].plot(df['PERM_mD'], df['DEPTH']); ax[3].set_xlabel('PERM mD')
plt.suptitle('Niger Delta Well - Petrophysical Quicklook')
plt.show()


### Next: Export results for ML ranking in Notebook 02
This dataset will be used to label reservoir quality for CO2 storage.
Transferability note: Same Vsh/Phi/Perm equations apply to WCSB Basal Belly River sandstones at CaMI FRS with adjusted GR cutoffs.
